# 🐍 Clase 7 · El loop de simulación

> Poner el tiempo en marcha: recorrer los snapshots, enviar órdenes en cada paso y llevar la cuenta de caja, inventario y equity con PositionTracker.

**Hoy construyes:** Market: reproducir snapshots y ejecutar en el tiempo.

### Cómo funciona este cuaderno

1. Escribe tu respuesta en la celda de código.
2. Debajo hay una **✅ comprobación plegada**: ejecútala con `Shift+Enter` para validarte (despliégala si quieres ver el `assert`).
3. ¿Atascado? Abre **💡 Ver solución**.

**Núcleo:** los primeros (en clase) · **Si vamos bien:** el resto · **Más:** el cuaderno de auxiliares.

### 1. Cuenta los pasos

Recorre todo el mercado con un while y cuenta los snapshots en `steps`.

<sub>practicas: el loop step()</sub>

In [ ]:
from exchange import Market
m = Market.sample()
steps = 0

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert steps == 500
print('ok')

<details>
<summary>💡 Ver solución</summary>

```python
from exchange import Market
m = Market.sample()
steps = 0
while True:
    book = m.step()
    if book is None: break
    steps += 1
```

</details>

### 2. Ejecuta una orden en un paso

Avanza un paso y envía una market buy de 0.2 con `m.submit(...)`. Guarda `filled`.

<sub>practicas: submit</sub>

In [ ]:
from exchange import Market, Order, Side, OrderType
m = Market.sample()
m.step()
filled = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert abs(filled - 0.2) < 1e-9
print('ok')

<details>
<summary>💡 Ver solución</summary>

```python
fills = m.submit(Order('BTCUSDT', Side.BUY, 0.2, order_type=OrderType.MARKET))
filled = sum(f.size for f in fills)
```

</details>

### 3. Acumula posición en el tiempo

Cada 50 pasos compra 0.1 (market). Aplica los fills a un `PositionTracker`. Guarda `final_pos`.

<sub>practicas: loop + tracker</sub>

In [ ]:
from exchange import Market, Order, Side, OrderType, PositionTracker
m = Market.sample()
tracker = PositionTracker()
i = 0
final_pos = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert final_pos > 0, 'has comprado varias veces: posición larga'
print('ok  pos=%.2f' % final_pos)

<details>
<summary>💡 Ver solución</summary>

```python
i = 0
while True:
    book = m.step()
    if book is None: break
    if i % 50 == 0:
        for f in m.submit(Order('BTCUSDT', Side.BUY, 0.1, order_type=OrderType.MARKET)):
            tracker.apply_fill(f)
    i += 1
final_pos = tracker.position
```

</details>

### 4. Equity final

Continuando, marca el equity al último mid visto. Guarda `equity`.

<sub>practicas: marcar a mercado</sub>

In [ ]:
from exchange import Market, Order, Side, OrderType, PositionTracker
m = Market.sample()
tracker = PositionTracker()
last_mid = 0
i = 0
while True:
    book = m.step()
    if book is None: break
    last_mid = book.mid
    if i % 50 == 0:
        for f in m.submit(Order('BTCUSDT', Side.BUY, 0.1, order_type=OrderType.MARKET)):
            tracker.apply_fill(f)
    i += 1
equity = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert isinstance(equity, float)
print('ok  equity=%.2f' % equity)

<details>
<summary>💡 Ver solución</summary>

```python
equity = tracker.equity(last_mid)
```

</details>

## Cierre

Mercado = estado (libro) + dinámica (matching) + tiempo (el loop). Todo junto, ya simulas.

Si llegas al ejercicio 3 ya tienes el núcleo. Los siguientes y los auxiliares consolidan.

**Siguiente clase:** seguimos construyendo el motor sobre esta pieza.